# Bilecik Building Segmentation — Final Model (YOLOv8s-seg)

Training notebook used on Google Colab for the final model in `weights/best.pt`.

**Changes compared to the baseline (YOLOv8n-seg, default settings):**
1. Larger model: `yolov8s-seg` instead of `yolov8n-seg`
2. Optimizer: AdamW with a lower initial learning rate and a cosine LR schedule
3. Stronger on-the-fly augmentation: MixUp, Copy-Paste, rotation, scale and HSV changes

Outputs were cleared before publishing.

In [ ]:
import os
import zipfile
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')
print("Drive mounted. Searching for the dataset zip...")

# 2. Find the dataset zip (file name contains 'Bilecik')
dataset_zip = None
for root, dirs, files in os.walk('/content/drive/MyDrive/'):
    for file in files:
        if "Bilecik" in file and file.endswith(".zip"):
            dataset_zip = os.path.join(root, file)
            break
    if dataset_zip:
        break

# 3. Extract it
if dataset_zip:
    print(f"Found: {dataset_zip}")
    os.makedirs("/content/Bilecik_Veri", exist_ok=True)
    with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall("/content/Bilecik_Veri")
    print("Dataset extracted.")
else:
    print("ERROR: dataset zip not found in Google Drive.")

In [ ]:
import yaml
import glob

# Locate data.yaml inside the extracted folder
yaml_file = glob.glob("/content/Bilecik_Veri/**/data.yaml", recursive=True)[0]

with open(yaml_file, 'r') as file:
    data = yaml.safe_load(file)

# Rewrite split paths as absolute Colab paths
data['train'] = yaml_file.replace('data.yaml', 'train/images')
data['val'] = yaml_file.replace('data.yaml', 'valid/images')
if 'test' in data:
    data['test'] = yaml_file.replace('data.yaml', 'test/images')

with open(yaml_file, 'w') as file:
    yaml.dump(data, file)

print(f"data.yaml updated. Train path: {data['train']}")

In [ ]:
!pip install -q ultralytics

from ultralytics import YOLO
import glob

# Change 1: use the 'small' model instead of 'nano' (more capacity)
model = YOLO('yolov8s-seg.pt')

yaml_path = glob.glob("/content/Bilecik_Veri/**/data.yaml", recursive=True)[0]

results = model.train(
    data=yaml_path,
    epochs=300,
    patience=0,         # early stopping disabled
    imgsz=640,
    plots=True,

    # Change 2: optimizer and learning-rate schedule
    optimizer='AdamW',
    lr0=0.001,          # lower initial learning rate
    cos_lr=True,        # cosine annealing

    # Change 3: stronger on-the-fly augmentation
    degrees=15.0,       # random rotation up to ±15°
    scale=0.4,          # random zoom in/out up to 40%
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,          # blend two images with 20% probability
    copy_paste=0.2,     # paste building instances onto other images
    hsv_s=0.3,          # saturation jitter
    hsv_v=0.3           # brightness jitter
)